# Six Sigma DMAIC — Phase 3: Analyze (分析阶段)

> **数据源**: `SigmaFlow/templates/DMAIC_03_Analyze_Template.xlsx`
> **核心工具 (全面覆盖 Flex 010~017 培训教材)**:
> 1. **抽样理论与中心极限定理演示 (CLT & Sampling)**: 总体偏态分布 vs N=5, N=30 样本均值模拟出图
> 2. **点图示范 (Dotplot)**: 小样本离散与聚集形态评估
> 3. **区间估计与区间图示范 (Interval Estimation & Interval Plot)**: 均值 95% 置信区间
> 4. **单样本假设检验**: 1-Sample t-Test ($\mu = \mu_0$) 与 1-Variance $\chi^2$ 检验
> 5. **双样本与配对检验**: 2-Sample t-Test (等方差/异方差)、方差齐性 F-Test、Paired t-Test (配对 t 检验)
> 6. **离散型数据假设检验**: 2-Proportions 比例检验、列联表卡方独立性检验 ($\chi^2$ Contingency Analysis) 与分类柱状图
> 7. **多变量分析 (Multi-Vari Chart)**: 件内、件间、时变三分量分析
> 8. **单因素方差分析与箱线图 (One-Way ANOVA & Boxplot)**: 机差比较与 Bartlett 方差齐性检验
> 9. **过程自变量相关性与多元回归 (Correlation & Regression)**
> 10. **过程 FMEA 风险分析 (RPN 评估表)**


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd(), Path.cwd() / "SigmaFlow", Path.cwd().parent] if (p / "minitab_dmaic_visuals.py").exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from IPython.display import display, HTML, Image

import importlib
import minitab_dmaic_visuals as mv
importlib.reload(mv)
mv.apply_minitab_theme()

TEMPLATE_PATH = PROJECT_ROOT / "templates" / "DMAIC_03_Analyze_Template.xlsx"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loading Analyze Template from: {TEMPLATE_PATH}")


## 3.1 抽样理论与中心极限定理演示 (CLT & Sampling Demonstration)
读取 `Sampling_CLT_Data` 工作表，演示偏态/指数总体随着样本量 $N=5$ 和 $N=30$ 抽样，样本均值分布迅速逼近正态分布（中心极限定理核心要义）。


In [ ]:
df_clt = pd.read_excel(TEMPLATE_PATH, sheet_name="Sampling_CLT_Data")

out_clt_img = FIG_DIR / "phase3_clt_simulation.png"
mv.plot_clt_simulation(
    df_clt["Sample_Means_N5"].dropna().values,
    df_clt["Sample_Means_N30"].dropna().values,
    population_data=df_clt["Population_Raw"].dropna().values,
    title="Central Limit Theorem Demonstration (Minitab Style)",
    output_path=str(out_clt_img)
)
display(Image(filename=str(out_clt_img)))


## 3.2 单变量分布形态分析 (Dotplot, Histogram & Boxplot 示范)
读取 `Hypothesis_1Sample` 工作表，针对三项关键质量特性 (CTQ) 连续数据分别绘制 Minitab 三大经典单变量形态图表并输出全景统计卡片：
- **Dimension_mm** (工件尺寸，标称 50.00 mm): 点图 (Dotplot)、正态拟合直方图 (Histogram) 及箱线图 (Boxplot)
- **Weight_g** (工件重量，标称 125.00 g): 正态拟合直方图 (Histogram) 及箱线图 (Boxplot)
- **Hardness_HRC** (材料洛氏硬度，标称 58.50 HRC): 正态拟合直方图 (Histogram) 及箱线图 (Boxplot)
- **Descriptive Statistics 表卡片**: 输出三变量样本量 N、均值 Mean、标准差 StDev、四分位数 (Q1, Median, Q3) 及 IQR


In [ ]:
df_1s = pd.read_excel(TEMPLATE_PATH, sheet_name="Hypothesis_1Sample")
vals_1s = df_1s["Dimension_mm"].dropna().values
vals_wt = df_1s["Weight_g"].dropna().values
vals_hd = df_1s["Hardness_HRC"].dropna().values

out_dotplot_img = FIG_DIR / "phase3_dotplot.png"
out_hist_img = FIG_DIR / "phase3_histogram.png"
out_box_img = FIG_DIR / "phase3_boxplot.png"

out_hist_wt_img = FIG_DIR / "phase3_histogram_weight.png"
out_box_wt_img = FIG_DIR / "phase3_boxplot_weight.png"

out_hist_hd_img = FIG_DIR / "phase3_histogram_hardness.png"
out_box_hd_img = FIG_DIR / "phase3_boxplot_hardness.png"

out_summary_table_img = FIG_DIR / "phase3_descriptive_stats_table.png"

# 1. Dimension_mm: 点图、直方图、箱线图
mv.plot_dotplot(
    vals_1s,
    title="Dotplot of Dimension_mm (Minitab Style)",
    xlabel="Dimension (mm)",
    output_path=str(out_dotplot_img)
)
mv.plot_histogram(
    vals_1s,
    title="Histogram of Dimension_mm (with Normal Fit)",
    xlabel="Dimension (mm)",
    ylabel="Density / Relative Frequency",
    output_path=str(out_hist_img)
)
mv.plot_single_boxplot(
    vals_1s,
    label="Dimension_mm",
    title="Boxplot of Dimension_mm (Minitab Style)",
    ylabel="Dimension (mm)",
    output_path=str(out_box_img)
)

# 2. Weight_g: 直方图、箱线图
mv.plot_histogram(
    vals_wt,
    title="Histogram of Weight_g (with Normal Fit)",
    xlabel="Weight (g)",
    ylabel="Density / Relative Frequency",
    output_path=str(out_hist_wt_img)
)
mv.plot_single_boxplot(
    vals_wt,
    label="Weight_g",
    title="Boxplot of Weight_g (Minitab Style)",
    ylabel="Weight (g)",
    output_path=str(out_box_wt_img)
)

# 3. Hardness_HRC: 直方图、箱线图
mv.plot_histogram(
    vals_hd,
    title="Histogram of Hardness_HRC (with Normal Fit)",
    xlabel="Hardness (HRC)",
    ylabel="Density / Relative Frequency",
    output_path=str(out_hist_hd_img)
)
mv.plot_single_boxplot(
    vals_hd,
    label="Hardness_HRC",
    title="Boxplot of Hardness_HRC (Minitab Style)",
    ylabel="Hardness (HRC)",
    output_path=str(out_box_hd_img)
)

# 4. 汇总三列数据的 Minitab 描述性统计全景表卡片
summary_rows = []
for col, unit, series in [("Dimension_mm", "mm", vals_1s), ("Weight_g", "g", vals_wt), ("Hardness_HRC", "HRC", vals_hd)]:
    q1 = np.percentile(series, 25)
    q3 = np.percentile(series, 75)
    summary_rows.append({
        "Variable": f"{col} ({unit})",
        "N": str(len(series)),
        "Mean": f"{series.mean():.3f}",
        "StDev": f"{series.std(ddof=1):.3f}",
        "Min": f"{series.min():.3f}",
        "Q1": f"{q1:.3f}",
        "Median": f"{np.median(series):.3f}",
        "Q3": f"{q3:.3f}",
        "IQR": f"{q3 - q1:.3f}",
        "Max": f"{series.max():.3f}"
    })

sum_df = pd.DataFrame(summary_rows)
mv.render_table_card(
    sum_df,
    title="Descriptive Statistics: Dimension_mm, Weight_g, Hardness_HRC",
    output_path=str(out_summary_table_img)
)

# 依次呈现统计表与图形
display(Image(filename=str(out_summary_table_img)))

print("=== [CTQ 1] Dimension_mm (尺寸) ===")
display(Image(filename=str(out_dotplot_img)))
display(Image(filename=str(out_hist_img)))
display(Image(filename=str(out_box_img)))

print("=== [CTQ 2] Weight_g (重量) ===")
display(Image(filename=str(out_hist_wt_img)))
display(Image(filename=str(out_box_wt_img)))

print("=== [CTQ 3] Hardness_HRC (硬度) ===")
display(Image(filename=str(out_hist_hd_img)))
display(Image(filename=str(out_box_hd_img)))


### 3.2.2 Minitab 图形化汇总报告 (Summary Report for Data / Graphical Summary)
六西格玛黑带/绿带项目最常用、信息密度最高的单组数据综合报告面板（对应 Minitab `Stat -> Basic Statistics -> Graphical Summary`）：
- **左侧图形区**:
  1. 正态拟合直方图 (Histogram with Normal Fit)
  2. 水平对齐箱线图 (Horizontal Boxplot)
  3. 均值与中位数 95% 置信区间图 (95% CI for Mean & Median)
- **右侧统计区**:
  1. Anderson-Darling 正态性检验 (A-Squared & Stephens P-Value)
  2. 集中趋势与离散度 (Mean, StDev, Variance, N)
  3. 分布形态指标 (**Skewness 偏度** 与 **Kurtosis 峰度**)
  4. 五数概括 (Minimum, Q1, Median, Q3, Maximum)
  5. 均值、中位数与标准差各自对应的 95% 置信区间 (95% CI for Mean, Median, StDev)


In [ ]:
out_summary_report_img = FIG_DIR / "phase3_graphical_summary_dimension.png"

# 生成 Minitab 经典 Graphical Summary Report 综合汇总图
mv.plot_graphical_summary(
    vals_1s,
    var_name="Dimension_mm",
    title="Summary Report for Dimension_mm",
    output_path=str(out_summary_report_img)
)

display(Image(filename=str(out_summary_report_img)))


### 3.2.3 多列数据综合对比 (Multi-Column Boxplot & Histogram Comparison)
在六西格玛实战中，常常需要将**2列或多列数据放在同一张图中进行横向对比**（例如班次 Shift 1 vs Shift 2 vs Shift 3，或不同机台/批次尺寸）：
1. **多列并排对比箱线图 (Multiple Columns Boxplot)**：同一坐标轴并排展现各列箱体、中位数红线、均值标记 ($\oplus$) 与均值变化趋势连线。
2. **多列同刻度分面直方图 (Paneled Histogram - Same Scale)**：上下分面严格共享 X 轴范围与刻度，直观辨识均值漂移与方差离散。
3. **多列同图叠加直方图 (Overlay Histogram with Normal Fits)**：半透明叠加各列频数柱体与正态拟合曲线，评估重叠度。


In [ ]:
df_multi = pd.read_excel(TEMPLATE_PATH, sheet_name="Hypothesis_2Sample")
multi_data = {
    "Shift_1": df_multi["Shift_1_Dimension"].dropna().values,
    "Shift_2": df_multi["Shift_2_Dimension"].dropna().values,
    "Shift_3": df_multi["Shift_3_Dimension"].dropna().values,
}

out_multi_box_img = FIG_DIR / "phase3_multi_column_boxplot.png"
out_multi_hist_paneled_img = FIG_DIR / "phase3_multi_column_histogram_paneled.png"
out_multi_hist_overlay_img = FIG_DIR / "phase3_multi_column_histogram_overlay.png"

# 1. 多列并排箱线图
mv.plot_multi_column_boxplot(
    multi_data,
    title="Boxplot of Multiple Columns: Shift 1 vs Shift 2 vs Shift 3 (Minitab Style)",
    ylabel="Dimension (mm)",
    output_path=str(out_multi_box_img)
)

# 2. 多列同刻度分面直方图
mv.plot_multi_column_histogram_paneled(
    multi_data,
    title="Paneled Histogram of Multiple Columns (Same X-Scale Comparison)",
    xlabel="Dimension (mm)",
    output_path=str(out_multi_hist_paneled_img)
)

# 3. 多列同图叠加直方图
mv.plot_multi_column_histogram_overlay(
    multi_data,
    title="Overlay Histogram of Multiple Columns (with Normal Fits)",
    xlabel="Dimension (mm)",
    output_path=str(out_multi_hist_overlay_img)
)

display(Image(filename=str(out_multi_box_img)))
display(Image(filename=str(out_multi_hist_paneled_img)))
display(Image(filename=str(out_multi_hist_overlay_img)))


### 3.2.4 Minitab 图形菜单全谱系展示 (Complete Minitab Graph Menu Suite)
除了直方图、点图与箱线图之外，全面实现并输出 Minitab **Graph** 菜单下的所有经典统计图形：
1. **Scatterplot (散点图)**: 探查 X 与 Y 连续变量的相关性，叠加最小二乘回归直线及方程与 $R^2$ 统计窗格。
2. **Matrix Plot (矩阵散点图)**: 针对多个 CTQ 特征生成两两成对散点矩阵，对角线呈现变量分布直方图。
3. **Bubble Plot (气泡图)**: 在二维散点图基础上引入第三维度（气泡面积大小）呈现三变量联合关系。
4. **Marginal Plot (边际图)**: 核心二维散点图并在 X 轴与 Y 轴边缘紧密附着直方图/箱线图，全景评估双变量分布。
5. **Stem-and-Leaf (茎叶图)**: 输出标准 Minitab 深度 (Depth)、茎 (Stem) 与叶 (Leaves) 字符统计卡片。
6. **Probability Plot (正态概率图)**: 真实正态概率纸非线性百分比坐标轴，95% 置信带与 Anderson-Darling 检验值。
7. **Empirical CDF (经验累积分布图)**: 阶梯式累积经验概率曲线叠加理论正态分布 CDF 曲线。
8. **Probability Distribution Plot (概率分布图)**: 钟形正态分布概率密度曲线，标出 $\alpha=0.05$ 双侧显著性拒绝域阴影与临界切点。
9. **Individual Value Plot (单值散点图)**: 展现各组别所有样本的微抖动原始散点、组均值实心菱形标记与变化趋势线。
10. **Line Plot (时序/顺序折线图)**: 按样本加工批次展示连续质量折线与均值基准线。


In [ ]:
# 1. 散点图 (Scatterplot with Fit)
out_scatter = FIG_DIR / "phase3_scatterplot.png"
mv.plot_scatterplot(
    vals_wt, vals_1s,
    title="Scatterplot of Dimension vs Weight (Minitab Style)",
    xlabel="Weight (g)", ylabel="Dimension (mm)",
    output_path=str(out_scatter)
)

# 2. 矩阵散点图 (Matrix Plot)
out_matrix = FIG_DIR / "phase3_matrix_plot.png"
mv.plot_matrix_plot(
    df_1s[["Dimension_mm", "Weight_g", "Hardness_HRC"]],
    title="Matrix Plot of CTQ Measurements (Minitab Style)",
    output_path=str(out_matrix)
)

# 3. 气泡图 (Bubble Plot)
out_bubble = FIG_DIR / "phase3_bubble_plot.png"
mv.plot_bubble_plot(
    vals_wt, vals_1s, size=vals_hd,
    title="Bubble Plot of Dimension vs Weight (Bubble Size = Hardness)",
    xlabel="Weight (g)", ylabel="Dimension (mm)", size_label="Hardness (HRC)",
    output_path=str(out_bubble)
)

# 4. 边际图 (Marginal Plot)
out_marginal = FIG_DIR / "phase3_marginal_plot.png"
mv.plot_marginal_plot(
    vals_wt, vals_1s,
    title="Marginal Plot of Dimension vs Weight (with Histograms)",
    xlabel="Weight (g)", ylabel="Dimension (mm)",
    marginal_type="histogram",
    output_path=str(out_marginal)
)

# 5. 茎叶图 (Stem-and-Leaf)
out_stem = FIG_DIR / "phase3_stem_and_leaf.png"
mv.plot_stem_and_leaf(
    vals_1s, var_name="Dimension_mm",
    output_path=str(out_stem)
)

# 6. 正态概率图 (Probability Plot)
out_prob = FIG_DIR / "phase3_probability_plot.png"
mv.plot_probability_plot(
    vals_1s,
    title="Probability Plot of Dimension_mm (Normal 95% CI)",
    xlabel="Dimension (mm)",
    output_path=str(out_prob)
)

# 7. 经验累积分布图 (Empirical CDF)
out_ecdf = FIG_DIR / "phase3_empirical_cdf.png"
mv.plot_empirical_cdf(
    vals_1s,
    title="Empirical CDF of Dimension_mm (with Normal CDF)",
    xlabel="Dimension (mm)",
    output_path=str(out_ecdf)
)

# 8. 概率分布图 (Probability Distribution Plot)
out_dist = FIG_DIR / "phase3_prob_dist_plot.png"
mv.plot_probability_distribution_plot(
    params=(vals_1s.mean(), vals_1s.std(ddof=1)),
    shade_type="two_tailed", alpha=0.05,
    title="Distribution Plot: Normal(μ=50.04, σ=0.38), α=0.05",
    xlabel="Dimension (mm)",
    output_path=str(out_dist)
)

# 9. 单值散点图 (Individual Value Plot)
out_indiv = FIG_DIR / "phase3_individual_value_plot.png"
mv.plot_individual_value_plot(
    multi_data,
    title="Individual Value Plot of Shifts (Shift 1, 2, 3)",
    ylabel="Dimension (mm)",
    output_path=str(out_indiv)
)

# 10. 时序折线图 (Line Plot)
out_line = FIG_DIR / "phase3_line_plot.png"
mv.plot_line_plot(
    vals_1s,
    title="Line Plot of Dimension_mm (Run Order)",
    xlabel="Sample Order", ylabel="Dimension (mm)",
    output_path=str(out_line)
)

# 依次展示 Minitab 菜单图形
print("--- [1] 散点图与边际图 ---")
display(Image(filename=str(out_scatter)))
display(Image(filename=str(out_marginal)))

print("--- [2] 矩阵散点图与气泡图 ---")
display(Image(filename=str(out_matrix)))
display(Image(filename=str(out_bubble)))

print("--- [3] 茎叶图与正态概率图 ---")
display(Image(filename=str(out_stem)))
display(Image(filename=str(out_prob)))

print("--- [4] 经验 CDF 与理论分布拒绝域图 ---")
display(Image(filename=str(out_ecdf)))
display(Image(filename=str(out_dist)))

print("--- [5] 单值图与时序折线图 ---")
display(Image(filename=str(out_indiv)))
display(Image(filename=str(out_line)))


### 3.2.5 特性要因图 / 鱼骨图 (Cause-and-Effect / Fishbone Diagram)
在六西格玛 Analyze（分析）阶段，项目团队利用 **特性要因图（石川图 / 鱼骨图，对应 Minitab: Stat -> Quality Tools -> Cause-and-Effect）** 全面梳理导致关键尺寸超差的潜在输入因子 ($X$)：
- **经典 6M 骨架分类**: 人 (Personnel)、机 (Machines)、料 (Materials)、法 (Methods)、测 (Measurement)、环 (Environment)
- **根本原因高亮**: 红色虚线框突出标注出后续假设检验 (ANOVA) 与回归分析重点验证锁定的核心因子（如交接班操作差异、进给速度偏高）。


In [ ]:
out_fishbone = FIG_DIR / "phase3_fishbone_diagram.png"
mv.plot_fishbone_diagram(
    effect_text="Dimension\nVariation\n(尺寸超差)",
    highlight_causes=["Shift Handover (交接班差异)", "Feed Rate High (进给量偏高)"],
    title="Cause-and-Effect (Fishbone) Diagram - Minitab Style",
    output_path=str(out_fishbone)
)
display(Image(filename=str(out_fishbone)))


## 3.3 区间估计与 Minitab 区间图示范 (Interval Plot with 95% CI & Means Table)
依据 Minitab 标准与方差分析合并标准差 (Pooled StDev)，计算各机台的均值及 95% 置信区间，绘制 Minitab 水平方向区间图，并输出对应的 Means 统计数据表卡片（包含分组、样本量 N、均值 Mean、标准差 StDev、95% CI 及 Pooled StDev）。


In [ ]:
df_anova = pd.read_excel(TEMPLATE_PATH, sheet_name="ANOVA_Data")
machines = sorted(df_anova["Machine"].unique())

ns = [len(df_anova[df_anova["Machine"] == m]["Dimension_mm"].dropna()) for m in machines]
means = [df_anova[df_anova["Machine"] == m]["Dimension_mm"].dropna().mean() for m in machines]
stdevs = [df_anova[df_anova["Machine"] == m]["Dimension_mm"].dropna().std(ddof=1) for m in machines]

# 计算合并标准差 Pooled StDev 与基于 ANOVA 误差自由度的 95% CI
total_n = sum(ns)
k = len(machines)
df_error = total_n - k
pooled_s = np.sqrt(sum((n - 1) * s**2 for n, s in zip(ns, stdevs)) / df_error)
t_crit = scipy_stats.t.ppf(0.975, df_error)

ci_lows = [m - t_crit * pooled_s / np.sqrt(n) for m, n in zip(means, ns)]
ci_highs = [m + t_crit * pooled_s / np.sqrt(n) for m, n in zip(means, ns)]
cis = [f"({lo:.3f}, {hi:.3f})" for lo, hi in zip(ci_lows, ci_highs)]

# 1. 绘制调正方向后的水平 Minitab 区间图 (Y轴为机台，X轴为数值)
out_interval_img = FIG_DIR / "phase3_interval_plot.png"
mv.plot_interval_plot(
    means, ci_lows, ci_highs, labels=machines,
    title="Interval Plot of Dimension_mm (95% CI for the Mean)",
    xlabel="Mean Dimension (mm)",
    ylabel="Machine",
    orientation="horizontal",
    output_path=str(out_interval_img)
)

# 2. 生成标准 Minitab Means 统计数据表卡片图片 (完全复刻教材样式)
means_table = pd.DataFrame({
    "Machine": machines,
    "N": [str(n) for n in ns],
    "Mean": [f"{m:.3f}" for m in means],
    "StDev": [f"{s:.3f}" for s in stdevs],
    "95% CI": cis
})

out_means_table_img = FIG_DIR / "phase3_interval_means_table.png"
mv.render_table_card(
    means_table,
    title="Means (One-Way ANOVA Interval Estimation)",
    footer_lines=[f"Pooled StDev = {pooled_s:.5f}"],
    output_path=str(out_means_table_img)
)

display(Image(filename=str(out_interval_img)))
display(Image(filename=str(out_means_table_img)))


## 3.4 单样本假设检验 (1-Sample t-Test & 1-Variance Test)
检验生产尺寸均值是否显著偏离目标标称值 50.00 mm ($H_0: \mu = 50.00$ vs $H_1: \mu 
eq 50.00$)，输出检验数据表图片。


In [ ]:
target_mu = 50.00
t_stat_1s, p_val_1s = scipy_stats.ttest_1samp(vals_1s, target_mu)

# 1-Variance Chi-Square Test
s_sq = np.var(vals_1s, ddof=1)
target_sigma_sq = 0.40**2
chi2_stat_1s = (len(vals_1s) - 1) * s_sq / target_sigma_sq
p_val_chi2 = 2 * min(scipy_stats.chi2.cdf(chi2_stat_1s, df=len(vals_1s)-1), 1 - scipy_stats.chi2.cdf(chi2_stat_1s, df=len(vals_1s)-1))

test_1s_df = pd.DataFrame([
    {"Hypothesis Test": "1-Sample t-Test (Mean vs 50.00)", "N": str(len(vals_1s)), "Sample Mean": f"{vals_1s.mean():.4f}", "Test Stat": f"t = {t_stat_1s:.3f}", "P-Value": f"{p_val_1s:.4f}", "Conclusion": "No Significant Difference" if p_val_1s >= 0.05 else "Significant Drift"},
    {"Hypothesis Test": "1-Variance Chi2 Test (vs σ=0.40)", "N": str(len(vals_1s)), "Sample StDev": f"{vals_1s.std(ddof=1):.4f}", "Test Stat": f"χ² = {chi2_stat_1s:.3f}", "P-Value": f"{p_val_chi2:.4f}", "Conclusion": "Variance Matches Target" if p_val_chi2 >= 0.05 else "Variance Significantly Differs"}
])

out_1s_table_img = FIG_DIR / "phase3_1sample_test_table.png"
mv.render_table_card(
    test_1s_df,
    title="One-Sample Hypothesis Tests Summary Table",
    footer_lines=[f"Null Hypotheses: H0: μ = 50.00 mm, H0: σ = 0.40 mm at α = 0.05."],
    output_path=str(out_1s_table_img)
)
display(Image(filename=str(out_1s_table_img)))


## 3.5 双样本与配对假设检验 (2-Sample t-Test, F-Test, Paired t-Test)
1. **双样本检验 (Shift 1 vs Shift 2)**: 比较两班次均值与等方差 F 检验。
2. **配对 t 检验 (Paired t-Test)**: 同批 25 件工件在在线气动量规与实验室三坐标 (CMM) 测量读数的差值检验。


In [ ]:
df_2s = pd.read_excel(TEMPLATE_PATH, sheet_name="Hypothesis_2Sample")
s1 = df_2s["Shift_1_Dimension"].dropna()
s2 = df_2s["Shift_2_Dimension"].dropna()

t_stat_2s, p_val_2s = scipy_stats.ttest_ind(s1, s2, equal_var=False)
f_var_stat = s1.var(ddof=1) / s2.var(ddof=1)
p_val_fvar = 2 * min(scipy_stats.f.cdf(f_var_stat, len(s1)-1, len(s2)-1), 1 - scipy_stats.f.cdf(f_var_stat, len(s1)-1, len(s2)-1))

test_2s_df = pd.DataFrame([
    {"Group": "Shift 1", "N": str(len(s1)), "Mean": f"{s1.mean():.4f}", "StDev": f"{s1.std(ddof=1):.4f}", "Test Stat": "-", "P-Value": "-"},
    {"Group": "Shift 2", "N": str(len(s2)), "Mean": f"{s2.mean():.4f}", "StDev": f"{s2.std(ddof=1):.4f}", "Test Stat": "-", "P-Value": "-"},
    {"Group": "Difference (Shift1 - Shift2)", "N": "-", "Mean": f"{s1.mean() - s2.mean():.4f}", "StDev": "-", "Test Stat": f"t = {t_stat_2s:.3f}", "P-Value": f"{p_val_2s:.4f}"},
    {"Group": "Test for Equal Variances (F-Test)", "N": "-", "Mean": "-", "StDev": f"Ratio = {f_var_stat:.3f}", "Test Stat": f"F = {f_var_stat:.3f}", "P-Value": f"{p_val_fvar:.4f}"}
])

out_2s_img = FIG_DIR / "phase3_2sample_ttest_table.png"
mv.render_table_card(
    test_2s_df,
    title="Two-Sample T-Test and Equal Variances Test Table",
    footer_lines=[f"T-statistic = {t_stat_2s:.3f}, p = {p_val_2s:.4f}; Equal Variance F p = {p_val_fvar:.4f}"],
    output_path=str(out_2s_img)
)
display(Image(filename=str(out_2s_img)))

# Paired t-Test
df_paired = pd.read_excel(TEMPLATE_PATH, sheet_name="Hypothesis_Paired")
t_paired, p_paired = scipy_stats.ttest_rel(df_paired["Online_Gage_mm"], df_paired["Lab_CMM_mm"])

paired_df = pd.DataFrame([
    {"Measurement Method": "Online Gage (在线量规)", "N": str(len(df_paired)), "Mean": f"{df_paired['Online_Gage_mm'].mean():.4f}", "StDev": f"{df_paired['Online_Gage_mm'].std():.4f}"},
    {"Measurement Method": "Lab CMM (实验室三坐标)", "N": str(len(df_paired)), "Mean": f"{df_paired['Lab_CMM_mm'].mean():.4f}", "StDev": f"{df_paired['Lab_CMM_mm'].std():.4f}"},
    {"Measurement Method": "Paired Difference (差值)", "N": str(len(df_paired)), "Mean": f"{df_paired['Difference_mm'].mean():.4f}", "StDev": f"{df_paired['Difference_mm'].std():.4f}"}
])

out_paired_img = FIG_DIR / "phase3_paired_ttest_table.png"
mv.render_table_card(
    paired_df,
    title="Paired T-Test: Online Gage versus Lab CMM",
    footer_lines=[f"T-statistic = {t_paired:.3f}, P-Value = {p_paired:.4f}", "Conclusion: No significant measurement system bias between tools."],
    output_path=str(out_paired_img)
)
display(Image(filename=str(out_paired_img)))


## 3.6 离散型数据检验与卡方分析 (2-Proportions & Chi-Square Contingency Test)
1. **双比率检验 (2-Proportions Test)**: 比较两条生产线的不良品率是否有统计学显著差异。
2. **卡方独立性检验 (Chi-Square Test of Independence)**: 检验机床 (Machine 1-4) 与不良类型是否相关。


In [ ]:
# 2-Proportions Test
from statsmodels.stats.proportion import proportions_ztest
df_prop = pd.read_excel(TEMPLATE_PATH, sheet_name="Hypothesis_2Proportion")
counts = df_prop["Defective_Count"].values
nobs = df_prop["Inspected_Count"].values
z_stat, p_prop = proportions_ztest(counts, nobs)

prop_summary = pd.DataFrame([
    {"Production Line": df_prop.loc[0, "Line"], "Inspected": str(nobs[0]), "Defects": str(counts[0]), "Rate": f"{counts[0]/nobs[0]*100:.2f}%"},
    {"Production Line": df_prop.loc[1, "Line"], "Inspected": str(nobs[1]), "Defects": str(counts[1]), "Rate": f"{counts[1]/nobs[1]*100:.2f}%"},
    {"Production Line": "Difference (Line A - Line B)", "Inspected": "-", "Defects": f"Z = {z_stat:.3f}", "Rate": f"p = {p_prop:.4f} (显著)" if p_prop < 0.05 else f"p = {p_prop:.4f}"}
])

out_prop_img = FIG_DIR / "phase3_2proportions_test_table.png"
mv.render_table_card(
    prop_summary,
    title="Test for Two Proportions Table (Line A vs Line B Defect Rate)",
    footer_lines=[f"Z-Value = {z_stat:.3f}, P-Value = {p_prop:.4f}; Line B has significantly higher defect rate."],
    output_path=str(out_prop_img)
)
display(Image(filename=str(out_prop_img)))

# Chi-Square Contingency Table
df_chi = pd.read_excel(TEMPLATE_PATH, sheet_name="ChiSquare_Contingency").set_index("Machine")
chi2_stat, p_chi2, dof_chi2, expected = scipy_stats.chi2_contingency(df_chi)

out_chi_chart = FIG_DIR / "phase3_chisquare_contingency_chart.png"
mv.plot_contingency_bar_chart(
    df_chi,
    title="Chi-Square Contingency Bar Chart (Machine vs Defect Type)",
    xlabel="Machine",
    ylabel="Defect Count",
    output_path=str(out_chi_chart)
)
display(Image(filename=str(out_chi_chart)))

chi_table_card = pd.DataFrame({
    "Machine": df_chi.index,
    "Oversize": [str(v) for v in df_chi["Defect_Oversize"]],
    "Undersize": [str(v) for v in df_chi["Defect_Undersize"]],
    "Roughness": [str(v) for v in df_chi["Defect_Roughness"]],
    "Total": [str(v) for v in df_chi.sum(axis=1)]
})

out_chi_table_img = FIG_DIR / "phase3_chisquare_table.png"
mv.render_table_card(
    chi_table_card,
    title="Contingency Table of Defect Modes across Machines",
    footer_lines=[f"Pearson Chi-Square = {chi2_stat:.3f}, DF = {dof_chi2}, P-Value = {p_chi2:.4f}", "Machine and defect distribution are statistically dependent (p < 0.05)."],
    output_path=str(out_chi_table_img)
)
display(Image(filename=str(out_chi_table_img)))


## 3.7 多变量图分析 (Multi-Vari Chart)
读取 `Multi_Vari_Data` 工作表，将变异来源拆解为：**件内变异 (Within-piece)**、**件间变异 (Piece-to-piece)** 和 **时变变异 (Time-to-time)**。


In [ ]:
df_mvari = pd.read_excel(TEMPLATE_PATH, sheet_name="Multi_Vari_Data")
out_mv_img = FIG_DIR / "phase3_multi_vari_chart.png"

fig_mv = mv.plot_multi_vari_chart(
    df_mvari,
    time_col="Time_Period",
    piece_col="Piece_ID",
    value_col="Dimension_mm",
    title="Multi-Vari Chart for dimension_mm across Time Periods",
    output_path=str(out_mv_img)
)
display(Image(filename=str(out_mv_img)))


## 3.8 单因素方差分析与箱线图示范 (One-Way ANOVA & Boxplot)
读取 `ANOVA_Data` 工作表，绘制机差箱线图并输出 ANOVA 方差分析数据表图片。


In [ ]:
df_anova = pd.read_excel(TEMPLATE_PATH, sheet_name="ANOVA_Data")
out_box_img = FIG_DIR / "phase3_anova_machines.png"

fig_box = mv.plot_anova_boxplot(
    df_anova,
    factor_col="Machine",
    response_col="Dimension_mm",
    title="Boxplot of Dimension_mm by Machine (One-Way ANOVA)",
    output_path=str(out_box_img)
)

import statsmodels.api as sm
from statsmodels.formula.api import ols
anova_fit = ols('Dimension_mm ~ C(Machine)', data=df_anova).fit()
anova_tab = sm.stats.anova_lm(anova_fit, typ=2)

anova_df = pd.DataFrame({
    "Source": ["C(Machine)", "Error (Residual)", "Total"],
    "DF": [str(int(anova_tab.loc['C(Machine)', 'df'])), str(int(anova_tab.loc['Residual', 'df'])), str(int(anova_tab['df'].sum()))],
    "SS": [f"{anova_tab.loc['C(Machine)', 'sum_sq']:.4f}", f"{anova_tab.loc['Residual', 'sum_sq']:.4f}", f"{anova_tab['sum_sq'].sum():.4f}"],
    "MS": [f"{anova_tab.loc['C(Machine)', 'sum_sq']/anova_tab.loc['C(Machine)', 'df']:.4f}", f"{anova_tab.loc['Residual', 'sum_sq']/anova_tab.loc['Residual', 'df']:.4f}", ""],
    "F": [f"{anova_tab.loc['C(Machine)', 'F']:.3f}", "", ""],
    "P": [f"{anova_tab.loc['C(Machine)', 'PR(>F)']:.4f}", "", ""]
})

out_anova_img = FIG_DIR / "phase3_anova_table.png"
mv.render_table_card(
    anova_df,
    title="One-Way ANOVA: Dimension_mm versus Machine",
    footer_lines=[f"F-Value = {anova_tab.loc['C(Machine)', 'F']:.3f}, P-Value = {anova_tab.loc['C(Machine)', 'PR(>F)']:.4f}", "Machine differences are statistically significant (p < 0.05)."],
    output_path=str(out_anova_img)
)

display(Image(filename=str(out_box_img)))
display(Image(filename=str(out_anova_img)))


## 3.9 工艺过程参数相关性与回归分析 (Correlation & Regression)
读取 `Process_Factors` 工作表，输出相关性矩阵热力图及多元线性回归系数表图片。


In [ ]:
df_factors = pd.read_excel(TEMPLATE_PATH, sheet_name="Process_Factors")
out_corr_img = FIG_DIR / "phase3_factors_correlation.png"

fig_corr = mv.plot_correlation_heatmap(
    df_factors[["temperature_C", "pressure_bar", "machine_speed_rpm", "coolant_ph", "dimension_mm"]],
    title="Correlation Matrix of Process Factors",
    output_path=str(out_corr_img)
)

reg_model = ols('dimension_mm ~ temperature_C + pressure_bar', data=df_factors).fit()

reg_df = pd.DataFrame({
    "Term": reg_model.params.index,
    "Coef": [f"{v:.5f}" for v in reg_model.params.values],
    "SE Coef": [f"{v:.5f}" for v in reg_model.bse.values],
    "T-Value": [f"{v:.3f}" for v in reg_model.tvalues.values],
    "P-Value": [f"{v:.4f}" for v in reg_model.pvalues.values]
})

out_reg_img = FIG_DIR / "phase3_regression_table.png"
mv.render_table_card(
    reg_df,
    title="Regression Coefficients Table: dimension_mm vs Temp & Pressure",
    footer_lines=[f"R-sq = {reg_model.rsquared*100:.2f}%, R-sq(adj) = {reg_model.rsquared_adj*100:.2f}%", f"S = {np.sqrt(reg_model.mse_resid):.4f}"],
    output_path=str(out_reg_img)
)

display(Image(filename=str(out_corr_img)))
display(Image(filename=str(out_reg_img)))


## 3.10 潜在失效模式与后果分析 (FMEA RPN 排查表图片)
读取 `FMEA` 工作表，生成风险优先数 (RPN) 排查分析数据表图片。


In [ ]:
df_fmea = pd.read_excel(TEMPLATE_PATH, sheet_name="FMEA")
out_fmea_img = FIG_DIR / "phase3_fmea_table.png"

fmea_disp = df_fmea[["Process_Step", "Failure_Mode", "Severity_S", "Potential_Cause", "Occurrence_O", "Detection_D", "RPN", "Action_Owner"]]

mv.render_table_card(
    fmea_disp.sort_values(by="RPN", ascending=False),
    title="Failure Mode and Effect Analysis (FMEA RPN Ranking Table)",
    footer_lines=["Critical Risk (RPN > 200): Spindle heat thermal expansion requires active oil-cooling control."],
    output_path=str(out_fmea_img),
    figsize=(10.5, 6.0)
)
display(Image(filename=str(out_fmea_img)))


## 3.11 根本原因统计验证 (Root Cause Validation: Operator versus Accuracy %)
> **实现原理**: 严格按照六西格玛黑带实战评审标准面板排版（ANOVA 方差分析表、Means 均值置信区间表与 Pooled StDev、Factor Information 表、带均值连线及离群值的 Boxplot、结论与 P-value 判定徽章）。
> - 读取 `Root_Cause_Operator` 工作表（6名操作员 Alex, Grace, Janet, Luke, Patricia, Wilson 共 152 次测试数据）。
> - 检验操作员对加工准确率 (Accuracy %) 是否存在统计学显著影响。
> - 检验结果：$F = 2.30, P = 0.048 < 0.05$，在 95% 置信水平下证实操作员是导致准确率波动的核心根本原因！


In [ ]:
df_root = pd.read_excel(TEMPLATE_PATH, sheet_name="Root_Cause_Operator")
out_root_img = FIG_DIR / "phase3_root_cause_validation.png"

mv.plot_root_cause_validation_report(
    df_root,
    factor_col="Operator",
    response_col="Accuracy_Pct",
    title_main="ROOT CAUSE 1-ACCURACY- STATISTICAL ANALYSIS",
    subtitle="ROOT CAUSE VALIDATION: OPERATOR",
    output_path=str(out_root_img)
)
display(Image(filename=str(out_root_img)))


### 📌 Analyze 阶段总结
1. **全面统计方法落地**: 抽样分析与中心极限定理、点图、区间图、单样本/双样本/配对t检验、方差齐性检验、比例与卡方独立性检验全覆盖。
2. **根本原因统计验证**: 针对关键影响因子（如操作员技能与机床系统）完成 ANOVA 闭环统计验证（$p = 0.048$），确证根本原因成立。
3. **所有图表与分析数据表图片完整生成归档**。
